# Voicers TTS vs Whisper STT Comparison

Generate audio with voicers, transcribe with Whisper, compare what was said vs what was heard.

In [ ]:
import subprocess, os

# Use the installed binary
VOICERS = "voice"

test_phrases = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog",
    "How are you doing today?",
    "Welcome to the voice synthesis engine",
    "She sells seashells by the seashore",
]

output_dir = "/tmp/voicers_stt_test"
os.makedirs(output_dir, exist_ok=True)

env = os.environ.copy()
env["PATH"] = "/opt/homebrew/bin:" + env.get("PATH", "")

for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    result = subprocess.run(
        [VOICERS, "-o", wav_path, phrase],
        capture_output=True,
        text=True,
        env=env,
    )
    status = "ok" if result.returncode == 0 else "FAIL"
    print(f"[{i}] {status} | '{phrase}' -> {wav_path}")
    for line in result.stderr.strip().split("\n"):
        if "chunk" in line.lower():
            print(f"    {line.strip()}")
    if result.returncode != 0:
        print(f"    {result.stderr[-300:]}")

print(f"\nDone: {len(test_phrases)} files in {output_dir}")

In [ ]:
import mlx_whisper

WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo"

results = []
for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    transcription = mlx_whisper.transcribe(wav_path, path_or_hf_repo=WHISPER_MODEL)
    heard = transcription["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    results.append((phrase, heard, match))
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

In [ ]:
# Test: is the sibilant issue G2P or vocoder?
import mlx_whisper

WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo"

files = {
    "seashells (G2P)": "/tmp/seashells.wav",
    "seashells (raw phonemes)": "/tmp/seashells_raw.wav",
    "this is a test (raw)": "/tmp/test_simple.wav",
}

for label, path in files.items():
    result = mlx_whisper.transcribe(path, path_or_hf_repo=WHISPER_MODEL)
    print(f"{label}:")
    print(f"  Heard: {result['text'].strip()}")
    print()

In [ ]:
# Is the garbling from G2P tokens or from something in the CLI text path?
files2 = {
    "G2P (via --text)": "/tmp/seashells.wav",
    "G2P phonemes (via --phonemes)": "/tmp/seashells_g2p_raw.wav",
    "Raw phonemes (no stress/collapse)": "/tmp/seashells_raw.wav",
}

for label, path in files2.items():
    result = mlx_whisper.transcribe(path, path_or_hf_repo=WHISPER_MODEL)
    print(f"{label}:")
    print(f"  Heard: {result['text'].strip()}")
    print()

In [ ]:
# Check if random seed causes inconsistency
runs = {
    "run 1": "/tmp/seashells_run1.wav",
    "run 2": "/tmp/seashells_run2.wav",
    "run 3": "/tmp/seashells_run3.wav",
}

for label, path in runs.items():
    result = mlx_whisper.transcribe(path, path_or_hf_repo=WHISPER_MODEL)
    print(f"{label}: {result['text'].strip()}")

In [ ]:
# Test consistency with fixed seed
print("=== Consistency test (same phonemes, 3 runs) ===")
for i in range(1, 4):
    result = mlx_whisper.transcribe(
        f"/tmp/seashells_seed{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    print(f"  run {i}: {result['text'].strip()}")

print("\n=== Full comparison (with fixed seed) ===")
test_phrases = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog",
    "How are you doing today?",
    "Welcome to the voice synthesis engine",
    "She sells seashells by the seashore",
]
for i, phrase in enumerate(test_phrases):
    result = mlx_whisper.transcribe(
        f"/tmp/voicers_stt_test/phrase_{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    heard = result["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

In [ ]:
# Test with zero phase initialization
print("=== Consistency test (zero phase, 3 runs) ===")
for i in range(1, 4):
    result = mlx_whisper.transcribe(
        f"/tmp/seashells_zero{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    print(f"  run {i}: {result['text'].strip()}")

print("\n=== Full comparison (zero phase) ===")
for i, phrase in enumerate(test_phrases):
    result = mlx_whisper.transcribe(
        f"/tmp/voicers_stt_test/phrase_{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    heard = result["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

In [ ]:
# Test with eval mode (dropout disabled) + zero phase
print("=== Consistency test (eval mode, 3 runs) ===")
for i in range(1, 4):
    result = mlx_whisper.transcribe(
        f"/tmp/seashells_eval{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    print(f"  run {i}: {result['text'].strip()}")

print("\n=== Full comparison (eval mode) ===")
for i, phrase in enumerate(test_phrases):
    result = mlx_whisper.transcribe(
        f"/tmp/voicers_stt_test/phrase_{i}.wav", path_or_hf_repo=WHISPER_MODEL
    )
    heard = result["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()